In [0]:
from pyspark.sql import functions as f  
from pyspark.sql import types as t         
from datetime import datetime   
import logging        
from config import ROUTES, PipelineConfig  


## CVM - Fundos de Investimento, Classes e Subclasses de Cotas

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
hoje = datetime.today()

# 0 = Segunda-feira, 6 = Domingo
if hoje.weekday() in [0, 6]: 
    log.info("Sem ingestão de Cadastros nas segundas e domingos.")
    dbutils.notebook.exit("Sucesso: Fora da janela de atualização da CVM")

In [0]:
# config
BASE_URL  = "https://dados.cvm.gov.br/dados/FI/CAD/DADOS/registro_fundo_classe.zip"
RAW_PATH  = f"{ROUTES.RAW_PATH}/cvm_registros/" 
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

### 1. Verificando os arquivos

Segundo o Site da CVM a atualização dos arquivos ocorre de terça a sábado, às 08:00h, com a posição cadastral dos fundos até as 23:59h do dia anterior. Sendo assim conseguimos economizar no processamento nesses dias 

In [0]:
# Extair dados do arquivo ZIP e inserir na camada RAW
PipelineConfig.baixar_e_extrair_zip(url=BASE_URL, raw_path=RAW_PATH)

In [0]:
# Schema da tabela: workspace.case_spark_cvm.registro_classe_cvm
SCHEMA_CLASSE = t.StructType([
    t.StructField("ID_Registro_Fundo", t.StringType(), True),
    t.StructField("ID_Registro_Classe", t.StringType(), True),
    t.StructField("CNPJ_Classe", t.StringType(), True),
    t.StructField("Codigo_CVM", t.StringType(), True),
    t.StructField("Data_Registro", t.StringType(), True),
    t.StructField("Data_Constituicao", t.StringType(), True),
    t.StructField("Data_Inicio", t.StringType(), True),
    t.StructField("Tipo_Classe", t.StringType(), True),
    t.StructField("Denominacao_Social", t.StringType(), True),
    t.StructField("Situacao", t.StringType(), True),
    t.StructField("Data_Inicio_Situacao", t.StringType(), True),
    t.StructField("Classificacao", t.StringType(), True),
    t.StructField("Indicador_Desempenho", t.StringType(), True),
    t.StructField("Classe_Cotas", t.StringType(), True),
    t.StructField("Classificacao_Anbima", t.StringType(), True),
    t.StructField("Tributacao_Longo_Prazo", t.StringType(), True),
    t.StructField("Entidade_Investimento", t.StringType(), True),
    t.StructField("Permitido_Aplicacao_CemPorCento_Exterior", t.StringType(), True),
    t.StructField("Classe_ESG", t.StringType(), True),
    t.StructField("Forma_Condominio", t.StringType(), True),
    t.StructField("Exclusivo", t.StringType(), True),
    t.StructField("Publico_Alvo", t.StringType(), True),
    t.StructField("Patrimonio_Liquido", t.StringType(), True),
    t.StructField("Data_Patrimonio_Liquido", t.StringType(), True),
    t.StructField("CNPJ_Auditor", t.StringType(), True),
    t.StructField("Auditor", t.StringType(), True),
    t.StructField("CNPJ_Custodiante", t.StringType(), True),
    t.StructField("Custodiante", t.StringType(), True),
    t.StructField("CNPJ_Controlador", t.StringType(), True),
    t.StructField("Controlador", t.StringType(), True),
])

# Schema da tabela: workspace.case_spark_cvm.registro_fundo_cvm
SCHEMA_FUNDO = t.StructType([
    t.StructField("ID_Registro_Fundo", t.StringType(), True),
    t.StructField("CNPJ_Fundo", t.StringType(), True),
    t.StructField("Codigo_CVM", t.StringType(), True),
    t.StructField("Data_Registro", t.StringType(), True),
    t.StructField("Data_Constituicao", t.StringType(), True),
    t.StructField("Tipo_Fundo", t.StringType(), True),
    t.StructField("Denominacao_Social", t.StringType(), True),
    t.StructField("Data_Cancelamento", t.StringType(), True),
    t.StructField("Situacao", t.StringType(), True),
    t.StructField("Data_Inicio_Situacao", t.StringType(), True),
    t.StructField("Data_Adaptacao_RCVM175", t.StringType(), True),
    t.StructField("Data_Inicio_Exercicio_Social", t.StringType(), True),
    t.StructField("Data_Fim_Exercicio_Social", t.StringType(), True),
    t.StructField("Patrimonio_Liquido", t.StringType(), True),
    t.StructField("Data_Patrimonio_Liquido", t.StringType(), True),
    t.StructField("Diretor", t.StringType(), True),
    t.StructField("CNPJ_Administrador", t.StringType(), True),
    t.StructField("Administrador", t.StringType(), True),
    t.StructField("Tipo_Pessoa_Gestor", t.StringType(), True),
    t.StructField("CPF_CNPJ_Gestor", t.StringType(), True),
    t.StructField("Gestor", t.StringType(), True),
])

# Schema da tabela: workspace.case_spark_cvm.registro_subclasse_cvm
SCHEMA_SUBCLASSE = t.StructType([
    t.StructField("ID_Registro_Classe", t.StringType(), True),
    t.StructField("ID_Subclasse", t.StringType(), True),
    t.StructField("Codigo_CVM", t.StringType(), True),
    t.StructField("Data_Constituicao", t.StringType(), True),
    t.StructField("Data_Inicio", t.StringType(), True),
    t.StructField("Denominacao_Social", t.StringType(), True),
    t.StructField("Situacao", t.StringType(), True),
    t.StructField("Data_Inicio_Situacao", t.StringType(), True),
    t.StructField("Forma_Condominio", t.StringType(), True),
    t.StructField("Exclusivo", t.StringType(), True),
    t.StructField("Publico_Alvo", t.StringType(), True),
    t.StructField("Previdenciario", t.StringType(), True),
    t.StructField("Exclusivo_INR", t.StringType(), True),
    t.StructField("Exclusivo_Previdencia_Complementar", t.StringType(), True),
])


### 2. Salvar em camada Bronze Particionada

In [0]:
# 1. Lista todos os arquivos CSV dentro do volume
arquivos_raw = [file.path for file in dbutils.fs.ls(RAW_PATH) if file.name.endswith('.csv')]

for arquivo in arquivos_raw:
    nomes_arquivos = arquivo.split('/')[-1].split('.csv')[0]

    # seleção do schema 
    if nomes_arquivos == "registro_classe":
        SCHEMA_ATUAL = SCHEMA_CLASSE
    elif nomes_arquivos == "registro_fundo":
        SCHEMA_ATUAL = SCHEMA_FUNDO
    elif nomes_arquivos == "registro_subclasse":
        SCHEMA_ATUAL = SCHEMA_SUBCLASSE
    else: 
        log.error(f"SCHEMA não encontrado")
        continue

    bronze_path = f"{ROUTES.TABLE_BASE}.bronze_{nomes_arquivos}_cvm"
    print(bronze_path)

    df = (spark.read
        .schema(SCHEMA_ATUAL)
        .option("encoding", "ISO-8859-1") 
        .option("sep", ";")
        .option("header", "true")
        .csv(arquivo)
    )


    # 6. Criação de metadados
    df = (df
        .withColumn("_source_url", f.lit(BASE_URL))
        .withColumn("_ingest_timestamp", f.current_timestamp())
        .withColumn("data_processamento", f.lit(DATA_PROC))
    )
    
    try:
        # 7. Escrita na Bronze 
        n = df.count()

        log.info(f"Escrevendo {n} linhas em Bronze ")

        (df.write 
            .mode("overwrite") 
            .option("replaceWhere", f"data_processamento = {DATA_PROC}") 
            .option("mergeSchema", "true")
            .option("delta.autoOptimize.optimizeWrite", "true")
            .option("delta.autoOptimize.autoCompact", "true")
            .partitionBy("data_processamento") 
            .format("delta") 
            .saveAsTable(bronze_path)
        )

        PipelineConfig.registrar_auditoria(
            spark, ROUTES.AUDIT_PATH, "bronze_raw_cvm_fundos_investimentos_classes_subclasse_cota",
            bronze_path, n, "SUCESSO", DATA_PROC
        )

    except Exception as e:
        PipelineConfig.registrar_auditoria(
            spark, ROUTES.AUDIT_PATH, "bronze_raw_cvm_fundos_investimentos_classes_subclasse_cota",
            bronze_path, 0, "FALHA", DATA_PROC, str(e)
        )
        raise



log.info("Processamento da RAW para BRONZE concluído com sucesso!")